# Linear Regression Health Costs Calculator

freeCodeCamp Machine Learning with Python project. Run all cells in order (Runtime → Run all).

In [ ]:
# Import libraries. You may or may not use all of these.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# Import data
!wget -q https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# Convert categorical columns to numbers
dataset['sex'] = (dataset['sex'] == 'male').astype(float)
dataset['smoker'] = (dataset['smoker'] == 'yes').astype(float)
dataset = pd.get_dummies(dataset, columns=['region'], dtype=float)

# Costs grow faster with age, and smokers with a high BMI cost far more,
# so give the model those relationships as extra columns
dataset['age_squared'] = dataset['age'] ** 2
dataset['obese'] = (dataset['bmi'] >= 30).astype(float)
dataset['smoker_bmi'] = dataset['smoker'] * dataset['bmi']
dataset['smoker_obese'] = dataset['smoker'] * dataset['obese']

dataset = dataset.astype('float32')
dataset.tail()

In [ ]:
# 80% train / 20% test
train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)

train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

In [ ]:
normalizer = layers.Normalization()
normalizer.adapt(np.array(train_dataset))

model = keras.Sequential([
    keras.Input(shape=(train_dataset.shape[1],)),
    normalizer,
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
    # Expenses are in the tens of thousands; scaling the output keeps the weights small
    layers.Rescaling(10000),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss='mae',
    metrics=['mae', 'mse'])

history = model.fit(
    train_dataset, train_labels,
    epochs=500,
    validation_split=0.2,
    verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)])

print(f"Trained for {len(history.history['loss'])} epochs, best validation MAE: {min(history.history['val_loss']):.2f}")

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)